<a href="https://colab.research.google.com/github/uranovnass/Machine-Learning/blob/main/randomforest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

# Загрузка данных
# Путь к файлу внутри папки sample_data
df = pd.read_csv("sample_data/o_churn_synthetic.csv")

X = df.drop(columns=["churn", "subscriber_code"])
y = df["churn"]

# Разбиение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Функция для предобработки данных
def make_prep():
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
        remainder="passthrough"
    )

# Вывод размеров получившихся выборок
print("train:", X_train.shape, " test:", X_test.shape)


train: (2400, 11)  test: (600, 11)


In [6]:
pip install xgboost


In [7]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def fit_and_report(name, pipe):
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    sec = time.perf_counter() - t0

    f1_tr = f1_score(y_train, pipe.predict(X_train))
    f1_te = f1_score(y_test, pipe.predict(X_test))
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])

    print(f"{name:<14} F1 train={f1_tr:.3f}  F1 test={f1_te:.3f}  "
          f"AUC test={auc:.3f}  {sec:.2f} c")



In [9]:
from xgboost import XGBClassifier

xgb = make_pipeline(make_prep(), XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42))

fit_and_report("XGBoost", xgb)



XGBoost        F1 train=0.789  F1 test=0.574  AUC test=0.779  0.28 c


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(make_prep(), RandomForestClassifier(n_estimators=100, random_state=42))
fit_and_report("СЛучайный лес",rf)

СЛучайный лес  F1 train=1.000  F1 test=0.556  AUC test=0.752  0.78 c


In [11]:
X_tr,X_val,y_te,y_val = train_test_split(
    X_train,y_train,test_size=0.2, stratify=y_train,random_state=42
)
p= make_prep().fit(X_tr)
Xtr,Xval = p.transform(X_tr), p.transform(X_val)
print("строк:",len(X_tr),len(X_val),len(X_test),
"признаки после jne hot:",Xtr.shape[1])


строк: 1920 480 600 признаки после jne hot: 22


In [20]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# 1. Генерируем искусственные данные (замените на свои, если они у вас есть)
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# 2. Делим данные на обучающую (train) и валидационную (val) выборки
Xtr, Xval, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Код со слайда ---

es = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=4,
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
)

es.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)

print("лучшая итерация:", es.best_iteration, "logloss на val:", round(es.best_score, 4))


лучшая итерация: 108 logloss на val: 0.2948


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaled = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
    remainder=StandardScaler()
)

models = {
    "Логрегрессия": make_pipeline(scaled, LogisticRegression(max_iter=1000)),
    "Случайный лес": make_pipeline(make_prep(), RandomForestClassifier(
        n_estimators=100, random_state=42)),
    "XGBoost": make_pipeline(make_prep(), XGBClassifier(
        n_estimators=127, learning_rate=0.05, max_depth=3, random_state=42)),
}

for name, pipe in models.items():
    fit_and_report(name, pipe)

Логрегрессия   F1 train=0.594  F1 test=0.611  AUC test=0.806  0.10 c
Случайный лес  F1 train=1.000  F1 test=0.556  AUC test=0.752  1.91 c
XGBoost        F1 train=0.645  F1 test=0.586  AUC test=0.792  0.51 c
